In [0]:
from pyspark.sql import functions as F

customer_scope = spark.table(
    "workspace.kkbox.customer_scope"
)

members_clean = spark.table(
    "workspace.kkbox.members_clean"
)

transaction_features_ml = spark.table(
    "workspace.kkbox.transaction_features_ml"
)

customer_features_final = (
    customer_scope
    .join(
        members_clean,
        on="msno",
        how="left"
    )
    .join(
        transaction_features_ml,
        on="msno",
        how="left"
    )
)

display(customer_features_final.limit(10))

In [0]:
customer_features_validation = (
    customer_features_final
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("msno").alias("unique_customers"),
        F.sum(
            F.when(F.col("is_churn").isNull(), 1).otherwise(0)
        ).alias("null_churn_labels"),
        F.sum(
            F.when(F.col("transaction_count").isNull(), 1).otherwise(0)
        ).alias("null_transaction_features"),
        F.sum(
            F.when(F.col("age").isNull(), 1).otherwise(0)
        ).alias("null_age"),
        F.sum(
            F.when(F.col("gender").isNull(), 1).otherwise(0)
        ).alias("null_gender"),
        F.sum(
            F.when(F.col("registered_via").isNull(), 1).otherwise(0)
        ).alias("null_registered_via")
    )
)

display(customer_features_validation)

In [0]:
customer_features_ml = (
    customer_features_final
    .withColumn(
        "age",
        F.coalesce(F.col("age"), F.lit(-1))
    )
    .withColumn(
        "city",
        F.coalesce(F.col("city"), F.lit("unknown"))
    )
    .withColumn(
        "registered_via",
        F.coalesce(
            F.col("registered_via").cast("string"),
            F.lit("unknown")
        )
    )
    .withColumn(
        "registration_date",
        F.coalesce(
            F.col("registration_date"),
            F.lit("unknown")
        )
    )
)

In [0]:
customer_features_ml = (
    customer_features_final
    .withColumn(
        "age",
        F.coalesce(F.col("age"), F.lit(-1))
    )
    .withColumn(
        "city",
        F.coalesce(
            F.col("city").cast("string"),
            F.lit("unknown")
        )
    )
    .withColumn(
        "registered_via",
        F.coalesce(
            F.col("registered_via").cast("string"),
            F.lit("unknown")
        )
    )
    .withColumn(
        "registration_date",
        F.coalesce(
            F.col("registration_date").cast("string"),
            F.lit("unknown")
        )
    )
)

In [0]:
ml_dataset_validation = (
    customer_features_ml
    .agg(
        F.count("*").alias("total_rows"),
        F.countDistinct("msno").alias("unique_customers"),

        F.sum(
            F.when(F.col("is_churn").isNull(), 1).otherwise(0)
        ).alias("null_churn_labels"),

        F.sum(
            F.when(F.col("age").isNull(), 1).otherwise(0)
        ).alias("null_age"),

        F.sum(
            F.when(F.col("city").isNull(), 1).otherwise(0)
        ).alias("null_city"),

        F.sum(
            F.when(F.col("registered_via").isNull(), 1).otherwise(0)
        ).alias("null_registered_via"),

        F.sum(
            F.when(F.col("registration_date").isNull(), 1).otherwise(0)
        ).alias("null_registration_date")
    )
)

display(ml_dataset_validation)

In [0]:
display(
    customer_features_ml
    .groupBy("is_churn")
    .count()
    .orderBy("is_churn")
)

In [0]:
customer_features_ml.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.kkbox.customer_features_ml")

In [0]:
spark.table("workspace.kkbox.customer_features_ml").count()

In [0]:
customer_features_ml.printSchema()

In [0]:
display(
    customer_features_ml.select(
        "msno",
        "is_churn",
        "age",
        "city",
        "gender",
        "registered_via",
        "registration_date",
        "transaction_count",
        "total_amount_paid",
        "avg_amount_paid",
        "total_plan_price",
        "avg_plan_days",
        "auto_renew_count",
        "cancel_count",
        "latest_transaction_date",
        "latest_expiry_date"
    ).limit(10)
)

In [0]:
customer_features_ml.printSchema()